In [ ]:
from datasets import load_dataset, Dataset, DatasetDict
from collections import defaultdict
import requests
import matplotlib.pyplot as plt
import pandas as pd
import json
import pathlib
from collections import Counter
import re

## Load SmolDoc

We will now load all SmolDoc datasets from the `google/smol` dataset on Hugging Face Datasets, and cross-reference them with our factuality annotations.

In [ ]:
from utils import list_smoldoc_configs, save_dataset_dict

dataset="google/smol"
smoldoc_configs = list_smoldoc_configs(dataset)
print(f"{len(smoldoc_configs)} SmolDoc configs")

We load all the SmolDoc configs into a dictionary of datasets.

In [ ]:
from utils import get_or_build_smoldoc

datasets_dict = get_or_build_smoldoc(
    dataset_name=dataset,
    configs=smoldoc_configs,
    save_path="data/smoldoc_datasets",
    overwrite=False  # Set to True if you want to force re-download
)

Simple statistics: how many unique topic ids are there across all SmolDoc datasets?

In [ ]:
# find all unique topic ids across all langauges
all_topic_ids = set()
for cfg, ds in datasets_dict.items():
    topic_ids = set(ds['id'])
    all_topic_ids.update(topic_ids)
print(f"Total unique topic ids across all SmolDoc topics: {len(all_topic_ids)}")

In [ ]:
# List all columns across configs
feature_map = defaultdict(set)
for cfg, ds in datasets_dict.items():
    feature_map[cfg].update(ds.column_names)
# Global union of features
all_features = sorted(set().union(*feature_map.values()))
print("\n=== All possible features across all configs ===")
print(all_features)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Compute topic counts
topic_counts = {cfg: len(ds) for cfg, ds in datasets_dict.items()}

# Build DataFrame
df_counts = (
    pd.DataFrame(list(topic_counts.items()), columns=["config", "num_topics"])
    .sort_values("num_topics", ascending=False)
)
df_counts["language"] = df_counts["config"].str.extract(r"smoldoc__([a-z]{2})")

# --- Plot setup: configs on X-axis, topics on Y-axis ---
plt.figure(figsize=(18, 8))  # wide to fit labels

bars = plt.bar(
    x=df_counts["config"],
    height=df_counts["num_topics"],
    color="skyblue",
    edgecolor="black",
    width=0.8
)

plt.ylabel("Number of Topics", fontsize=12)
plt.xlabel("SmolDoc Config", fontsize=12)
plt.title("Number of Topics per SmolDoc Config", fontsize=14, fontweight="bold")

# Rotate and space out config labels
plt.xticks(rotation=60, ha='right', fontsize=8)
plt.subplots_adjust(bottom=0.35)  # space for long config names

# Add value labels rotated vertically
for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2 + 0.2,
        height + 2,
        f"{int(height)}",
        ha="center",
        va="bottom",
        fontsize=8,
        rotation=45
    )

plt.tight_layout()
plt.show()

Now load the factuality annotations from JSON, and load it as a Dataset. We will then compare the annotated topic ids with those present in the English SmolDoc datasets.

In [ ]:
from utils import get_smoldoc_factuality, load_topic_ratings

JSON_PATH = "smoldoc-factuality-ratings.json"  # adjust if needed

data = get_smoldoc_factuality()
topic_ds = load_topic_ratings(json_data=data)
topic_ds

## Join factuality annotations to SmolDoc datasets

We have found out that there are not only English as source language datasets in SmolDoc, so we will only compare with those, to be consistent. Let us find all English SmolDoc configs, and collect their topic ids.

In [ ]:
english_cfgs = [c for c in datasets_dict if c.startswith("smoldoc__en_")]
english_topic_ids = set().union(*[set(datasets_dict[c]["id"]) for c in english_cfgs])

annot_topic_ids = set(x for x in topic_ds["topic_key"] if x is not None)

missing_in_annotations = sorted(english_topic_ids - annot_topic_ids)
extra_in_annotations_vs_english = sorted(annot_topic_ids - english_topic_ids)

print(f"# Annotation rows: {len(topic_ds)}")
print(f"# Unique annotated topic_ids: {len(annot_topic_ids)}")
print(f"# English topic_ids in datasets: {len(english_topic_ids)}")
print(f"# English topic_ids missing annotations: {len(missing_in_annotations)}")
print(f"# Annotated topic_ids not present in English: {len(extra_in_annotations_vs_english)}")

Now we will join the factuality annotations to all SmolDoc datasets (all languages), for those topic ids that are present in the annotations. This will allow us to analyze factuality across all languages.

In [ ]:
keep_cols = [
    'annotator_1_label', 'annotator_1_notes',
    'annotator_2_label', 'annotator_2_notes',
    'annotator_3_label', 'annotator_3_notes'
]

# Build lookup dict: topic_key -> annotations
annot_fields_by_id = {
    k: {col: v for col, v in zip(keep_cols, vals)}
    for k, *vals in zip(
        topic_ds["topic_key"],
        *[topic_ds[col] for col in keep_cols]
    )
}

def _join_annotations(batch):
    ids = batch["id"]
    # Vectorized per-column build
    return {
        col: [annot_fields_by_id[i][col] if i in annot_fields_by_id else None for i in ids]
        for col in keep_cols
    }

# Join all configs
fact_annot_ds = datasets_dict.map(_join_annotations, batched=True)


In [ ]:
save_dataset_dict(fact_annot_ds, "data/smoldoc_factuality_joined", overwrite=True)

Here is an example of one of the joined datasets with factuality annotations.

In [ ]:
example_cfg = 'smoldoc__en_sw'
fact_annot_ds[example_cfg]

In [ ]:
fact_annot_ds[example_cfg].features